# Travel Assistant — Мультиагентная система с RAG

## О проекте

Система помощи сотрудникам в организации командировок с использованием:
- **RAG** (Retrieval Augmented Generation) для поиска правил политики
- **Мультиагентов** (Поисковик, Аналитик, Бронировщик)
- **Локальной LLM** (заглушка или OpenRouter)
- **Векторной БД** (Chroma InMemory)

**Важно:** Все данные генерируются внутри ноутбука, дополнительные файлы не требуются.

## Раздел 1: Установка зависимостей

In [ ]:
!pip install -q langchain langgraph langchain-community chromadb sentence-transformers torch pandas openai

## Раздел 2: Конфигурация и вспомогательные функции

In [ ]:
import os
from typing import Dict, List, Optional, Any
import pandas as pd
from io import StringIO

# ==========================================
# ENV переменные (можно переопределить при необходимости)
# ==========================================

# LLM: предпочтительно OpenRouter, иначе заглушка
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "deepseek/deepseek-r1:free")

# Эмбеддер (используем лёгкую английскую модель, работает на CPU)
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

def print_config():
    print("\n" + "="*50)
    print("ТЕКУЩАЯ КОНФИГУРАЦИЯ")
    print("="*50)
    if OPENROUTER_API_KEY:
        print(f"LLM (OpenRouter): {OPENROUTER_MODEL}")
    else:
        print("LLM: DummyLLM (заглушка, т.к. нет OPENROUTER_API_KEY)")
    print(f"Эмбеддер: {EMBEDDING_MODEL}")
    print("="*50 + "\n")

print_config()

## Раздел 3: Генерация тестовых данных (без внешних файлов)

In [ ]:
# Создаём DataFrame с авиабилетами
tickets_data = """flight_number,airline,departure_city,arrival_city,departure_date,price,is_direct
SU100,Aeroflot,Москва,Санкт-Петербург,2025-06-01,12500,True
SU101,Aeroflot,Москва,Санкт-Петербург,2025-06-01,18900,False
SU200,Aeroflot,Москва,Казань,2025-06-02,8700,True
S7100,S7 Airlines,Москва,Новосибирск,2025-06-03,21500,True
SU300,Aeroflot,Москва,Сочи,2025-06-04,16200,True
U6100,Ural Airlines,Екатеринбург,Москва,2025-06-05,9900,True
"""
tickets_df = pd.read_csv(StringIO(tickets_data))
print(f"Загружено {len(tickets_df)} рейсов")

# Создаём текстовый документ политики
policy_text = """
Политика компании по командировкам:

1. Авиабилеты:
   - Максимальная стоимость билета в одну сторону — 50 000 рублей.
   - Эконом-класс является приоритетным.
   - Прямые рейсы предпочтительнее, чем с пересадками, если разница в цене не превышает 30%.

2. Проживание:
   - Максимальная стоимость отеля в России — 8 000 рублей за ночь.
   - Обязательно наличие Wi-Fi.
   - Завтрак приветствуется, но не обязателен.

3. Суточные:
   - По России — 2 500 рублей в день.
   - За рубежом — 5 000 рублей в день.

4. Трансфер:
   - Такси от/до аэропорта оплачивается при предъявлении чека.
"""

print(f"Сгенерирован текст политики ({len(policy_text)} символов)")

## Раздел 4: RAG-компоненты (эмбеддинги, чанкинг, векторное хранилище)

In [ ]:
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Загрузка модели эмбеддингов (легковесная, работает на CPU)
print("Загрузка модели эмбеддингов...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Модель эмбеддингов готова")

# Создаём объекты Document для политики и билетов
policy_doc = Document(page_content=policy_text, metadata={"source": "policy"})

ticket_docs = []
for _, row in tickets_df.iterrows():
    content = (f"Рейс {row['flight_number']} авиакомпании {row['airline']} "
               f"от {row['departure_city']} до {row['arrival_city']} "
               f"на {row['departure_date']} за {row['price']} руб. "
               f"{'Прямой' if row['is_direct'] else 'С пересадкой'}.")
    metadata = {
        "airline": row['airline'],
        "flight_number": row['flight_number'],
        "departure_city": row['departure_city'],
        "arrival_city": row['arrival_city'],
        "departure_date": row['departure_date'],
        "price": row['price'],
        "is_direct": 'Да' if row['is_direct'] else 'Нет',
        "source": "ticket"
    }
    ticket_docs.append(Document(page_content=content, metadata=metadata))

all_docs = [policy_doc] + ticket_docs
print(f"Всего документов: {len(all_docs)}")

# Чанкинг (только для политики, билеты остаются как есть)
# Разбиваем только политику на части, чтобы улучшить поиск
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

# Разбиваем политику на чанки, остальные документы оставляем как есть
chunks = text_splitter.split_documents([policy_doc]) + ticket_docs
print(f"Создано {len(chunks)} чанков (включая билеты)")

# Создаём векторное хранилище InMemory
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="travel_assistant",
    persist_directory=":memory:"
)
print("Векторное хранилище Chroma создано (InMemory)")

## Раздел 5: LLM-клиент (OpenRouter или заглушка)

In [ ]:
# Определяем LLM
if OPENROUTER_API_KEY:
    print(f"\nИспользуем OpenRouter: {OPENROUTER_MODEL}")
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
        model=OPENROUTER_MODEL,
        temperature=0.7
    )
    print("LLM через OpenRouter готов")
else:
    print("\nOPENROUTER_API_KEY не задан, используем DummyLLM (заглушка)")
    class DummyLLM:
        def invoke(self, prompt: str, **kwargs) -> str:
            return (f"[DummyLLM] Сгенерирован ответ на запрос: {prompt[:150]}...\n"
                    f"(Для реального ответа укажите OPENROUTER_API_KEY)")
        __call__ = invoke
    llm = DummyLLM()
    print("DummyLLM готов (имитирует ответы)")

# Тест LLM
print("\nТест LLM:")
test_response = llm.invoke("Привет! Ты — travel assistant. Напиши короткий приветственный ответ.")
print(test_response[:200], "...\n")

## Раздел 6: Агенты (PolicyExpert, TicketSearcher, BudgetAnalyst, HotelBooker)

In [ ]:
from typing import TypedDict, List, Dict, Any

class AgentState(TypedDict):
    """Состояние, передаваемое между агентами"""
    user_query: str
    extracted_params: Dict[str, Any]
    policy_context: str
    tickets: List[Dict[str, Any]]
    budget_check: Dict[str, Any]
    final_answer: str
    next_agent: str
    logs: List[str]

# ==========================================
# АГЕНТ 1: PolicyExpert (эксперт по политикам, RAG)
# ==========================================
class PolicyExpert:
    def __init__(self, vectorstore, embeddings, llm):
        self.vectorstore = vectorstore
        self.embeddings = embeddings
        self.llm = llm
    
    def invoke(self, state: AgentState) -> AgentState:
        query = state["user_query"]
        print(f"\n[PolicyExpert] Поиск политики по запросу: '{query[:80]}...'")
        # Ищем только документы с source=policy
        relevant_docs = self.vectorstore.similarity_search(
            query, k=3, filter={"source": "policy"}
        )
        policy_context = "\n\n".join([doc.page_content for doc in relevant_docs])
        print(f"Найдено {len(relevant_docs)} релевантных фрагментов политики")
        state["policy_context"] = policy_context
        state["next_agent"] = "TicketSearcher"
        state["logs"].append(f"PolicyExpert: найдено {len(relevant_docs)} фрагментов")
        return state

# ==========================================
# АГЕНТ 2: TicketSearcher (поиск билетов из CSV)
# ==========================================
class TicketSearcher:
    def __init__(self, tickets_df: pd.DataFrame):
        self.tickets_df = tickets_df
    
    def _extract_city(self, query: str, city_keywords: Dict[str, List[str]]) -> Optional[str]:
        query_lower = query.lower()
        for city, keywords in city_keywords.items():
            if any(kw in query_lower for kw in keywords):
                return city
        return None
    
    def invoke(self, state: AgentState) -> AgentState:
        query = state["user_query"].lower()
        
        # Словарь городов и их ключевых слов
        city_map = {
            "Москва": ["москв", "мск"],
            "Санкт-Петербург": ["питер", "петербург", "спб"],
            "Казань": ["казан"],
            "Новосибирск": ["новосибирск"],
            "Сочи": ["сочи"],
            "Екатеринбург": ["екатеринбург"]
        }
        
        departure = self._extract_city(query, city_map)
        arrival = self._extract_city(query, city_map)
        # Если город только один, считаем его пунктом прибытия
        if departure and not arrival:
            arrival = departure
            departure = None
        
        params = {
            "departure": departure,
            "arrival": arrival,
            "date": None  # для простоты не парсим дату
        }
        print(f"\n[TicketSearcher] Извлечённые параметры: {params}")
        
        # Фильтрация
        filtered = self.tickets_df
        if params["departure"]:
            filtered = filtered[filtered["departure_city"] == params["departure"]]
        if params["arrival"]:
            filtered = filtered[filtered["arrival_city"] == params["arrival"]]
        
        tickets = filtered.to_dict(orient="records")
        print(f"Найдено билетов: {len(tickets)}")
        
        state["extracted_params"] = params
        state["tickets"] = tickets
        state["next_agent"] = "BudgetAnalyst"
        state["logs"].append(f"TicketSearcher: найдено {len(tickets)} билетов")
        return state

# ==========================================
# АГЕНТ 3: BudgetAnalyst (проверка соответствия бюджету)
# ==========================================
class BudgetAnalyst:
    def __init__(self, llm):
        self.llm = llm
        self.limits = {
            "max_ticket_price": 50000,
            "daily_per_diem_russia": 2500,
            "daily_per_diem_abroad": 5000
        }
    
    def invoke(self, state: AgentState) -> AgentState:
        tickets = state["tickets"]
        if not tickets:
            print("\n[BudgetAnalyst] Нет билетов для проверки")
            state["budget_check"] = {"status": "error", "message": "Билеты не найдены"}
            state["next_agent"] = "HotelBooker"
            return state
        
        ticket = tickets[0]
        price = ticket["price"]
        max_limit = self.limits["max_ticket_price"]
        within = price <= max_limit
        
        print(f"\n[BudgetAnalyst] Билет {ticket['airline']} {ticket['flight_number']} стоимостью {price} руб.")
        print(f"Лимит: {max_limit} руб. -> {'Входит' if within else 'Превышает'}")
        
        state["budget_check"] = {
            "ticket_price": price,
            "max_limit": max_limit,
            "within_budget": within,
            "message": f"Билет за {price} руб. {'входит' if within else 'превышает'} лимит ({max_limit} руб.)"
        }
        state["next_agent"] = "HotelBooker"
        state["logs"].append(f"BudgetAnalyst: {state['budget_check']['message']}")
        return state

# ==========================================
# АГЕНТ 4: HotelBooker (подбор отеля на основе политики)
# ==========================================
class HotelBooker:
    def __init__(self):
        # База отелей по городам
        self.hotels = {
            "Москва": [
                {"name": "Бизнес-отель \"Аэростар\"", "price": 6500, "has_wifi": True, "has_breakfast": True},
                {"name": "Гостиница \"Космос\"", "price": 4800, "has_wifi": True, "has_breakfast": False},
                {"name": "Swissôtel Красные Холмы", "price": 12000, "has_wifi": True, "has_breakfast": True},
            ],
            "Санкт-Петербург": [
                {"name": "Отель \"Амбассадор\"", "price": 7200, "has_wifi": True, "has_breakfast": True},
                {"name": "Гостиница \"Москва\"", "price": 5500, "has_wifi": True, "has_breakfast": False},
            ],
            "Казань": [
                {"name": "Ramada Kazan", "price": 5900, "has_wifi": True, "has_breakfast": True},
            ],
            "Новосибирск": [
                {"name": "Marins Park Hotel", "price": 4300, "has_wifi": True, "has_breakfast": True},
            ],
            "Сочи": [
                {"name": "Отель \"Жемчужина\"", "price": 7800, "has_wifi": True, "has_breakfast": True},
            ],
        }
    
    def invoke(self, state: AgentState) -> AgentState:
        tickets = state.get("tickets", [])
        if not tickets:
            print("\n[HotelBooker] Нет информации о билетах, невозможно определить город")
            state["final_answer"] = "Не удалось подобрать отель: не указан город командировки."
            state["next_agent"] = "end"
            return state
        
        arrival_city = tickets[0].get("arrival_city", "Москва")
        print(f"\n[HotelBooker] Поиск отеля в городе: {arrival_city}")
        
        if arrival_city not in self.hotels:
            state["final_answer"] = f"К сожалению, в городе {arrival_city} нет отелей в нашей базе."
            state["next_agent"] = "end"
            return state
        
        available = self.hotels[arrival_city]
        # Фильтруем по политике: цена ≤ 8000 и Wi-Fi
        suitable = [h for h in available if h["price"] <= 8000 and h["has_wifi"]]
        if suitable:
            best = min(suitable, key=lambda x: x["price"])
            message = (f"Рекомендуемый отель: {best['name']}, {best['price']} руб./ночь, "
                       f"Wi-Fi: {'да' if best['has_wifi'] else 'нет'}, завтрак: {'да' if best['has_breakfast'] else 'нет'}")
        else:
            # Если ничего не подходит, берём самый дешёвый с Wi-Fi
            with_wifi = [h for h in available if h["has_wifi"]]
            if with_wifi:
                best = min(with_wifi, key=lambda x: x["price"])
                message = (f"Отель {best['name']} ({best['price']} руб./ночь) превышает бюджет, но имеет Wi-Fi. "
                           f"Требуется согласование с руководителем.")
            else:
                message = f"Нет подходящих отелей с Wi-Fi в городе {arrival_city}."
        
        print(f"{message}")
        state["best_hotel"] = best if 'best' in locals() else None
        state["final_answer"] = message
        state["next_agent"] = "end"
        state["logs"].append(f"HotelBooker: {message[:100]}")
        return state

## Раздел 7: Оркестратор (LangGraph)

In [ ]:
from langgraph.graph import StateGraph, END

class TravelAgentManager:
    """Оркестратор, управляющий последовательностью агентов"""
    def __init__(self, vectorstore, embeddings, llm, tickets_df):
        self.policy_expert = PolicyExpert(vectorstore, embeddings, llm)
        self.ticket_searcher = TicketSearcher(tickets_df)
        self.budget_analyst = BudgetAnalyst(llm)
        self.hotel_booker = HotelBooker()
    
    def create_graph(self):
        workflow = StateGraph(AgentState)
        
        workflow.add_node("PolicyExpert", self.policy_expert.invoke)
        workflow.add_node("TicketSearcher", self.ticket_searcher.invoke)
        workflow.add_node("BudgetAnalyst", self.budget_analyst.invoke)
        workflow.add_node("HotelBooker", self.hotel_booker.invoke)
        
        workflow.set_entry_point("PolicyExpert")
        
        # Условные переходы
        workflow.add_conditional_edges(
            "PolicyExpert",
            lambda x: x.get("next_agent", "TicketSearcher"),
            {"TicketSearcher": "TicketSearcher", END: END}
        )
        workflow.add_conditional_edges(
            "TicketSearcher",
            lambda x: x.get("next_agent", "BudgetAnalyst"),
            {"BudgetAnalyst": "BudgetAnalyst", END: END}
        )
        workflow.add_conditional_edges(
            "BudgetAnalyst",
            lambda x: x.get("next_agent", "HotelBooker"),
            {"HotelBooker": "HotelBooker", END: END}
        )
        workflow.add_conditional_edges(
            "HotelBooker",
            lambda x: x.get("next_agent", END),
            {END: END}
        )
        
        return workflow.compile()

# Создание оркестратора
manager = TravelAgentManager(vectorstore, embeddings, llm, tickets_df)
app = manager.create_graph()
print("\nОркестратор (TravelAgentManager) успешно создан!\n")

## Раздел 8: Демонстрация работы

In [ ]:
# Вспомогательная функция для запуска запроса
def run_query(user_query: str):
    initial_state = {
        "user_query": user_query,
        "extracted_params": {},
        "policy_context": "",
        "tickets": [],
        "budget_check": {},
        "final_answer": "",
        "next_agent": "PolicyExpert",
        "logs": []
    }
    result = app.invoke(initial_state)
    print("\n" + "="*60)
    print("ИТОГОВЫЙ ОТВЕТ:")
    print("="*60)
    print(result["final_answer"])
    if result["logs"]:
        print("\nЛог выполнения:")
        for log in result["logs"]:
            print(f"  • {log}")
    return result

# Пример 1: Поиск билетов Москва → Санкт-Петербург
print("\nПРИМЕР 1: Поиск билетов из Москвы в Питер")
run_query("Нужны билеты из Москвы в Санкт-Петербург на завтра")

# Пример 2: Вопрос о правилах
print("\n\nПРИМЕР 2: Какие правила по авиабилетам?")
run_query("Какие ограничения по стоимости авиабилетов в командировках?")

# Пример 3: Город, которого нет в базе отелей
print("\n\nПРИМЕР 3: Билет в Екатеринбург (нет отелей в базе)")
run_query("Нужен билет из Екатеринбурга в Москву")

## Завершение

Прототип мультиагентного travel-ассистента готов и работает без внешних файлов.

**Что можно улучшить:**
- Подключить реальный LLM через OpenRouter (укажите `OPENROUTER_API_KEY` в переменных окружения)
- Добавить парсинг дат и более сложную маршрутизацию
- Расширить базу отелей и билетов
- Интегрировать API бронирования

Удачи в разработке.